In [54]:
import numpy as np
import pandas as pd
from scipy.signal import correlate2d , convolve2d

In [55]:
alpha = 0.01
learning_rate = 0.01
epoch = 10

In [56]:
TRAIN_DATA_PATH = "../data/train.csv"
TEST_DATA_PATH = "../data/test.csv"

data = pd.read_csv(TRAIN_DATA_PATH)
data = np.array(data)

In [57]:
class Convolution:
    input_size = (0,0,0)
    filter_ze = 0
    filter_shape = (0,0,0)
    no_of_filters = 0
    input_data = None 

    def __init__(self, input_size , filter_size  , no_of_filters):
        input_height, input_width = input_size
        self.input_size = input_size
        self.filter_size = filter_size 
        self.no_of_filters = no_of_filters
        
        self.filter_shape = ( no_of_filters, filter_size , filter_size )
        self.output_size = (no_of_filters,input_height-self.filter_size+1 , input_width - self.filter_size+1 ) 
        self.filters = np.random.randn(*self.filter_shape)
        self.bias = np.zeros(self.output_size)
    def leaky_relu(self,z: np.array):
        return np.where(z > 0 , z , alpha * z)
    
    def forward_propagation(self ,data : np.matrix ):
        output = np.zeros(self.output_size)
        self.input_data = data
        for i in range(self.no_of_filters):
            output[i] = correlate2d(self.input_data, self.filters[i] , mode="valid")
        output = self.leaky_relu(output)
        return output  
    
    def back_propagation( self , dl_do : np.matrix) -> np.matrix :
        dl_dfilters = np.zeros_like(self.filters)
        dl_dinput = np.zeros_like(self.input_data)
        for i in range(self.no_of_filters):
           dl_dfilters[i] = correlate2d(self.input_data , dl_do[i] , mode="valid")
           dl_dinput = convolve2d(dl_do[i],self.filters[i] , mode= "full")
        self.filters -= learning_rate * dl_dfilters
        self.bias -= learning_rate * dl_do
        return dl_dinput
        

In [58]:
class MaxPooling :
    pool_size = 0
    no_of_filters = 0 
    output_size = ()
    def __init__(self , pool_size  ):
        self.pool_size = pool_size
    
    def forward_propgataion(self, featureMap: np):
        self.no_of_filters, self.height , self.width = featureMap.shape
        self.output_size = (self.no_of_filters, self.height // self.pool_size , self.width // self.pool_size)
        self.output = np.zeros(self.output_size)
        self.input_data = featureMap
        for i in range(self.no_of_filters): 
            for height in range(self.output_size[1]):
                for width in range(self.output_size[2]):
                    height_start = height * self.pool_size
                    width_start =  width * self.pool_size  

                    height_end = height_start + self.pool_size
                    width_end = width_start + self.pool_size 

                    self.output[i][height][width] = np.max(self.input_data[i , height_start : height_end , width_start : width_end])
        return self.output
        
    def back_propagation(self , dl_do: np.matrix) -> np.matrix:
        dl_dx = np.zeros_like(self.input_data) 
        
        for i in range(self.no_of_filters):
            for height in range(self.output_size[1]) :
                 for width in range(self.output_size[2]):
                    height_start = height * self.pool_size
                    width_start =  width * self.pool_size  

                    height_end = height_start + self.pool_size
                    width_end = width_start + self.pool_size 
                    patch = self.input_data[i , height_start : height_end , width_start: width_end]
                    mask = patch == np.max(patch)  

                    dl_dx[i , height_start: height_end , width_start : width_end ] = dl_do[i][height][width] * mask
        return dl_dx

In [59]:
class FNN:
    curr_neurons= 0
    input_size = 0
    weights:np.matrix = None
    bias: np.matrix = None
    def __init__(self , learning_rate : int , curr_neurons , input_size):
       self.curr_neurons = curr_neurons
       self.input_size = input_size 
       self.weights = np.random.randn(curr_neurons , input_size) 
       self.bias = np.random.randn(curr_neurons, 1)
       self.learning_rate = learning_rate
    
    def softmax(self, z:np.array):
        z = z - np.max(z) 
        exp = np.exp(z)
        result = exp/exp.sum()
        return result.reshape(-1,1)
    
    def leaky_relu(self, z: np.array):
        return np.where(z > 0 , z , alpha * z)
    
    def act_tanh(self, z : np.array):
        return np.tanh(z)
    
    def grad_tanh(self, z : np.array):
        return 1 - (np.tanh(z) ** 2)

    def grad_relu(self, z:np.array):
        return np.where(z > 0 , 1 , alpha)
    
    def softmax_derivative(self, s):
        return np.diagflat(s) - np.dot(s, s.T)
 
    def forward(self , input_data , act_func):
        self.input_data = input_data
        self.act_func = act_func
        self.Z = self.weights @ input_data + self.bias
        A = None
        if act_func == "relu":
            A =  self.leaky_relu(self.Z)
        elif act_func == "tanh":
            A = self.act_tanh(self.Z)
        else :
            A = self.softmax(self.Z)
        return A

    def backward(self , dl_dA : np.matrix):
        if self.act_func == "relu":
           dl_dz = dl_dA * self.grad_relu(self.Z) 
        elif self.act_func == "tanh":
            dl_dz = dl_dA * self.grad_tanh(self.Z)
        else:
            dl_dz = dl_dA 
        
        dl_dw = dl_dz @ self.input_data.T
        dl_db = dl_dz

        dl_dA_prev = self.weights.T @ dl_dz 

        self.weights -= self.learning_rate *  dl_dw
        self.bias -= self.learning_rate * dl_db
        return dl_dA_prev

In [60]:
def entropy_loss(y:np.array , y_hat):
    return -np.sum(y * np.log(y_hat + 1e-9)) 

def to_onehot(label, classes = 10):
    y = np.zeros(classes)
    y[label] = 1 
    return y

In [ ]:
train_data_raw = data[0:41000, 1:] / 255
train_label = data[0:41000,0:1]

validataion_data_raw = data[41000: , 1: ] /255
validation_label = data[41000: , 0:1]

# Reshape to 28x28
train_data = train_data_raw.reshape(-1, 28, 28)
validataion_data = validataion_data_raw.reshape(-1, 28, 28)

conv = Convolution((28, 28), 6, 1)
maxpooling = MaxPooling(2)
hidden_layer = FNN(learning_rate , 200 , 121 )
output_layer = FNN(learning_rate , 10 , 200)

for i in range(epoch):
    curr_loss = 0
    correct_pred = 0
    total_loss = 0
    for index, sample in enumerate(train_data):
        conv_out = conv.forward_propagation(sample)
        pool_out = maxpooling.forward_propgataion(conv_out)
        pool_out_flattend = pool_out.reshape(-1,1)
        hidden_out = hidden_layer.forward(pool_out_flattend, act_func="relu")
        output_out = output_layer.forward(hidden_out , act_func="softmax")

        class_actual = int(train_label[index][0])
        class_act_encoding = to_onehot(class_actual)

        curr_loss = entropy_loss(output_out , class_act_encoding.reshape(-1, 1))
        total_loss += curr_loss

        class_pred = np.argmax(output_out)

        if class_pred == class_actual:
            correct_pred += 1
        
        output_layer_back = output_layer.backward(output_out - class_act_encoding.reshape(-1, 1))
        hidden_layer_back = hidden_layer.backward(output_layer_back)
        hidden_layer_back_reshaped = hidden_layer_back.reshape(pool_out.shape)
        pool_layer_back = maxpooling.back_propagation(hidden_layer_back_reshaped)
        conv_layer_back = conv.back_propagation(pool_layer_back)

    average_loss = total_loss / len(train_data)
    accuracy = correct_pred / len(train_data) * 100.0
    print(f"Epoch {i + 1}/{epoch} - Loss: {average_loss:.4f} - Accuracy: {accuracy:.2f}%")

ValueError: could not broadcast input array from shape (18,18) into shape (6,6)

In [ ]:
def predict(data, label=None):
    predictions = []
    
    for sample in data:
        # Forward pass through the network
        conv_out = conv.forward_propagation(sample)
        pool_out = maxpooling.forward_propgataion(conv_out)
        pool_out_flattened = pool_out.reshape(-1, 1)
        hidden_out = hidden_layer.forward(pool_out_flattened, act_func="relu")
        output = output_layer.forward(hidden_out, act_func="softmax")
        
        # Get predicted class
        pred_class = np.argmax(output)
        predictions.append(pred_class)
    
    predictions = np.array(predictions)
    
    # Calculate accuracy only if labels are provided
    if label is not None:
        accuracy = np.mean(predictions == label.flatten()) * 100
        return predictions, accuracy
    else:
        return predictions


In [ ]:
# Validation with labels
print("\n--- Validation Results ---")
val_pred, val_acc = predict(validataion_data, validation_label)
print(f"Validation Accuracy: {val_acc:.2f}%")

# Test without labels
print("\n--- Test Results ---")
test_data_raw = np.array(pd.read_csv(TEST_DATA_PATH))
test_data_unnormalized = test_data_raw / 255  # Skip first column (ID), normalize
test_data = test_data_unnormalized.reshape(-1, 28, 28)  # Reshape to 28x28
test_pred = predict(test_data)  # No labels for test

# Save predictions to CSV
output_df = pd.DataFrame({
    'ImageId': range(1, len(test_pred) + 1),
    'Label': test_pred
})
output_df.to_csv('../predictions.csv', index=False)
print(f"Predictions saved to predictions.csv")